# V1-08: 기업 식별 보류·사용자 확인 정책 비교

V1은 비금융 OpenDART 공시기업을 상장·비상장 구분 없이 후보로 검색한다. 후보가 여러 개이거나 문자열 유사도 검색으로만 찾은 경우, 시스템이 1위를 자동 선택할지 아니면 사용자에게 후보를 보여 주고 확인할지를 비교한다.

## 비교 대상과 평가 기준

- **A. 최상위 후보 자동 선택**: 후보가 하나 이상이면 1위를 즉시 확정
- **B. 명확한 정확 일치만 확정**: 정확 일치 후보가 정확히 하나일 때만 확정하고, 복수 후보·유사 후보는 사용자 확인 요청

비교 지표는 Auto-resolution Success Rate, Coverage, Unsafe Selection Rate, 보류 정확도(Abstention Accuracy), Clarification Rate, p50/p95 지연시간이다. LLM은 사용하지 않으며, 후보 목록과 상태는 OpenDART 데이터·결정론적 규칙으로 만든다.

In [2]:
import io
from math import ceil
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process
import rapidfuzz

In [3]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

In [4]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes

print(f'RapidFuzz 버전: {rapidfuzz.__version__}')

RapidFuzz 버전: 3.14.5


## 1. 전체 기업코드 목록으로 후보 생성

상장 여부는 후보를 제거하는 조건이 아니라 후보에 함께 표시하는 메타데이터다. 각 후보에는 정식 법인명, `corp_code`, 상장 여부, 종목코드(있는 경우)를 남긴다.

In [5]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
print(f'전체 법인 레코드 수: {len(corp_records):,}')

전체 법인 레코드 수: 118,831


In [6]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(records_by_code.values()) for key, records_by_code in index.items()}

name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'정규화 이름 키 수: {len(name_keys):,}')

정규화 이름 키 수: 218,777


## 2. 개발용 정답 상태

정확한 공식명은 자동 확정이 허용되는 사례다. 오타·부분 표현·별칭·약칭·존재하지 않는 입력은 이번 정책에서 자동 확정하지 않는 것이 정답이다. 유사 후보의 품질은 07번의 Candidate Recall@K와 함께 해석한다.

In [7]:
TEST_CASES = [
    {'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930', '보류 정답': False},
    {'유형': '영문 대소문자', '질문 기업명': 'naver', '정답 종목코드': '035420', '보류 정답': False},
    {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '정답 종목코드': '005930', '보류 정답': True},
    {'유형': '부분 표현', '질문 기업명': '하이닉스', '정답 종목코드': '000660', '보류 정답': True},
    {'유형': '짧은 약칭·동명', '질문 기업명': '삼전', '정답 종목코드': None, '보류 정답': True},
    {'유형': '한글 별칭', '질문 기업명': '네이버', '정답 종목코드': '035420', '보류 정답': True},
    {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None, '보류 정답': True},
]

TEST_CASES

[{'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930', '보류 정답': False},
 {'유형': '영문 대소문자', '질문 기업명': 'naver', '정답 종목코드': '035420', '보류 정답': False},
 {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '정답 종목코드': '005930', '보류 정답': True},
 {'유형': '부분 표현', '질문 기업명': '하이닉스', '정답 종목코드': '000660', '보류 정답': True},
 {'유형': '짧은 약칭·동명', '질문 기업명': '삼전', '정답 종목코드': None, '보류 정답': True},
 {'유형': '한글 별칭', '질문 기업명': '네이버', '정답 종목코드': '035420', '보류 정답': True},
 {'유형': '존재하지 않는 기업',
  '질문 기업명': 'FinSight가상기업',
  '정답 종목코드': None,
  '보류 정답': True}]

## 3. 후보 생성과 확정 정책

후보 생성은 두 정책에서 동일하다. 차이는 후보가 나왔을 때 최종 `corp_code`를 확정하는 규칙이다. B의 `NEEDS_CONFIRMATION`은 다음 workflow에서 사용자에게 후보를 보여 주어야 한다는 상태이며, LLM이 기업을 고르는 상태가 아니다.

In [8]:
FUZZY_KEY_LIMIT = 20
RETURN_K = 3

def rank_records(records, score=100.0):
    return [
        (record, score)
        for record in sorted(
            records,
            key=lambda record: (bool(record['stock_code'].strip()), record['modify_date']),
            reverse=True,
        )
    ]

def find_candidates(query):
    exact_records = name_index.get(normalize_name(query), [])
    if exact_records:
        return rank_records(exact_records)[:RETURN_K], 'exact'

    scored_keys = process.extract(
        normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT
    )
    best_by_corp_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            previous = best_by_corp_code.get(record['corp_code'])
            if previous is None or score > previous[1]:
                best_by_corp_code[record['corp_code']] = (record, score)

    ranked = sorted(
        best_by_corp_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )
    return ranked[:RETURN_K], 'fuzzy'

def auto_top1_policy(candidates, source):
    if not candidates:
        return 'NOT_FOUND', None
    return 'RESOLVED', candidates[0][0]

def confirmation_policy(candidates, source):
    if not candidates:
        return 'NOT_FOUND', None
    if source == 'exact' and len(candidates) == 1:
        return 'RESOLVED', candidates[0][0]
    return 'NEEDS_CONFIRMATION', None

## 4. 결과와 지표 계산

각 후보에는 공식 법인명, `corp_code`, 상장 여부·종목코드, 유사도 점수를 출력한다. `NEEDS_CONFIRMATION`이면 이 목록을 사용자에게 보여 주고 선택을 받는 것이 다음 workflow의 계약이다.

In [9]:
WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

def candidate_view(record, score):
    stock_code = record['stock_code'].strip()
    return {
        '법인명': record['corp_name'],
        'corp_code': record['corp_code'],
        '상장 상태': '상장' if stock_code else '비상장',
        '종목코드': stock_code or None,
        '점수': round(score, 1),
    }

def evaluate_policy(policy_name, policy_function):
    rows = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            candidates, source = find_candidates(case['질문 기업명'])
            policy_function(candidates, source)

        case_latencies = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates, source = find_candidates(case['질문 기업명'])
            status, selected = policy_function(candidates, source)
            case_latencies.append((time.perf_counter() - started_at) * 1_000)

        rows.append({
            **case,
            '정책': policy_name,
            '후보 출처': source,
            '상태': status,
            '선택 종목코드': selected['stock_code'].strip() if selected else None,
            '후보': [candidate_view(record, score) for record, score in candidates],
        })
        latencies.extend(case_latencies)

    positive_rows = [row for row in rows if row['정답 종목코드']]
    resolved_rows = [row for row in rows if row['상태'] == 'RESOLVED']
    abstention_rows = [row for row in rows if row['보류 정답']]
    correct_auto_resolutions = sum(
        row['상태'] == 'RESOLVED' and row['선택 종목코드'] == row['정답 종목코드']
        for row in positive_rows
    )
    unsafe_selections = sum(
        row['선택 종목코드'] != row['정답 종목코드']
        for row in resolved_rows
    )

    summary = {
        '정책': policy_name,
        'Auto-resolution Success Rate': correct_auto_resolutions / len(positive_rows),
        'Coverage': sum(row['상태'] == 'RESOLVED' for row in positive_rows) / len(positive_rows),
        'Unsafe Selection Rate': unsafe_selections / len(resolved_rows) if resolved_rows else 0.0,
        'Abstention Accuracy': sum(row['상태'] != 'RESOLVED' for row in abstention_rows) / len(abstention_rows),
        'Clarification Rate': sum(row['상태'] == 'NEEDS_CONFIRMATION' for row in rows) / len(rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': sorted(latencies)[ceil(len(latencies) * 0.95) - 1],
    }
    return rows, summary

auto_rows, auto_summary = evaluate_policy('A. 최상위 후보 자동 선택', auto_top1_policy)
confirm_rows, confirm_summary = evaluate_policy('B. 명확한 정확 일치만 확정', confirmation_policy)

for row in auto_rows + confirm_rows:
    print({key: row[key] for key in ['정책', '유형', '질문 기업명', '후보 출처', '상태', '선택 종목코드', '후보']})

print(auto_summary)
print(confirm_summary)

{'정책': 'A. 최상위 후보 자동 선택', '유형': '정확 법인명', '질문 기업명': '삼성전자', '후보 출처': 'exact', '상태': 'RESOLVED', '선택 종목코드': '005930', '후보': [{'법인명': '삼성전자', 'corp_code': '00126380', '상장 상태': '상장', '종목코드': '005930', '점수': 100.0}]}
{'정책': 'A. 최상위 후보 자동 선택', '유형': '영문 대소문자', '질문 기업명': 'naver', '후보 출처': 'exact', '상태': 'RESOLVED', '선택 종목코드': '035420', '후보': [{'법인명': 'NAVER', 'corp_code': '00266961', '상장 상태': '상장', '종목코드': '035420', '점수': 100.0}]}
{'정책': 'A. 최상위 후보 자동 선택', '유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '후보 출처': 'fuzzy', '상태': 'RESOLVED', '선택 종목코드': '005930', '후보': [{'법인명': '삼성전자', 'corp_code': '00126380', '상장 상태': '상장', '종목코드': '005930', '점수': 75.0}, {'법인명': '삼성전기', 'corp_code': '00126371', '상장 상태': '상장', '종목코드': '009150', '점수': 75.0}, {'법인명': '삼성기전', 'corp_code': '01024478', '상장 상태': '비상장', '종목코드': None, '점수': 75.0}]}
{'정책': 'A. 최상위 후보 자동 선택', '유형': '부분 표현', '질문 기업명': '하이닉스', '후보 출처': 'fuzzy', '상태': 'RESOLVED', '선택 종목코드': '452400', '후보': [{'법인명': '이닉스', 'corp_code': '00338170', '상장 상태': '상장', '종목코드': '

## 해석 기준

- B가 Unsafe Selection Rate를 낮추고 보류 정확도를 높이면, 복수 후보·유사 후보의 기본 정책으로 채택할 근거가 된다.
- B의 Coverage와 Auto-resolution Success Rate가 낮아지는 것은 자동 확정을 줄인 직접 결과다. 사용자가 후보를 선택한 뒤의 최종 Entity Accuracy는 이 notebook에서 측정하지 않는다.
- 존재하지 않는 기업에도 유사 후보가 생기면 `NEEDS_CONFIRMATION`이 나올 수 있다. 점수 cutoff로 `NOT_FOUND`를 구분할지는 이 결과를 본 뒤 별도 후보 비교로 결정한다.